# HKA Error Galleries — Banded Case Review (OAI External Validation)

Companion to `hto_correction_angles_oai_external_validation.ipynb`.

This notebook re-runs the landmark model over the OAI full-limb cohort, computes HKA from
the predicted landmarks, compares it against the OAI clinical (OAISYS) HKA, and splits the
cohort into **three interactive galleries** by disagreement size:

| gallery | band | what lives there |
|---|---|---|
| **1 — gross** | `\|error\| > HKA_ERROR_THRESHOLD_DEG` (10°) | discrete, nameable failures |
| **2 — moderate** | `HKA_ERROR_MID_DEG` – `HKA_ERROR_THRESHOLD_DEG` (5–10°) | landmarks drifting; clinically material |
| **3 — close** | `\|error\| ≤ HKA_ERROR_MID_DEG` (5°) | the bulk of the cohort; working as intended |

Both cut points are constants. The bands use `lo < |error| ≤ hi`, so they are **mutually
exclusive and exhaustive** — every knee appears in exactly one gallery, which the
verification cell checks rather than assumes.

**Why galleries.** The aggregate numbers (MAE ≈ 1.4°, ICC ≈ 0.75) hide a long tail. The
useful question is not *how big is the tail* but *what kind of picture lives in it* — a
laterality/sign flip, a mis-detected femoral head, a cropped ankle, an implant, a duplicated
or rotated acquisition. Those are visually obvious and statistically invisible. The three
bands separate the failure modes, because a 25° case and a 6° case are almost never the
same problem.

**What you get**

- a self-contained inference sweep (nothing needs to be in memory from the other notebook)
- per-knee results cached to CSV so re-runs are cheap
- three `ipywidgets` galleries: paged thumbnail grid → click any tile for a full-size view
- overlays: the 12 predicted landmarks (colour-coded) and the hip–knee–ankle
  **mechanical axis** used to derive HKA, drawn on the offending leg
- a range slider on each, so you can re-cut at 7–8° or 15°+ without re-running inference

**Run order:** set the config cell → run top-to-bottom → the gallery appears near the end.

> The letterboxed 768 px canvas is the space the predicted coordinates live in, so every
> overlay is drawn there directly — no inverse-transform step, nothing to get wrong.

## Imports & shared configuration

Mirrors the validation notebook so the two stay in lockstep. Inference only — no
optimiser, scheduler, or augmentation.

In [1]:
import os
import io
import sys
import math
import glob
import random
import numpy as np
import pandas as pd
import torch
from PIL import Image
from functools import lru_cache

# figure objects are built directly (no pyplot) so the notebook's inline backend
# is left untouched and rendering stays thread-safe
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.lines import Line2D

for _ckd in ("CKD", "/tf/notebooks/CKD", os.path.join(os.getcwd(), "CKD")):
    if os.path.isdir(_ckd):
        sys.path.append(os.path.abspath(_ckd)); break
else:
    sys.path.append(os.path.abspath("CKD"))

from models import (
    Conformer_tiny_patch16_keypoint_half_heatmap,
    Conformer_small_patch16_keypoint_half_heatmap,
    Conformer_small_patch32_keypoint_half_heatmap,
    Conformer_base_patch16_keypoint_half_heatmap,
)
from utils import extract_coordinates

SEED          = 42
TARGET_SIZE   = 768
HEATMAP_SCALE = 0.5
MODEL_VARIANT = "small_p16"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware device registered: {device}")

GLOBAL_KEYPOINT_NAMES = [
    "femur_head_lh", "knee_inner_lh", "ost_point_lh",
    "knee_outer_lh", "ankle_inner_lh", "ankle_outer_lh",
    "femur_head_rh", "knee_inner_rh", "ost_point_rh",
    "knee_outer_rh", "ankle_inner_rh", "ankle_outer_rh",
]

LANDMARK_COLORS = {
    "femur_head_lh":  "darkgreen",  "knee_inner_lh":  "darkblue",
    "ost_point_lh":   "darkred",    "knee_outer_lh":  "darkviolet",
    "ankle_inner_lh": "darkorange", "ankle_outer_lh": "teal",
    "femur_head_rh":  "lightgreen", "knee_inner_rh":  "lightblue",
    "ost_point_rh":   "lightcoral", "knee_outer_rh":  "plum",
    "ankle_inner_rh": "sandybrown", "ankle_outer_rh": "paleturquoise",
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)

/usr/local/lib/python3.11/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.11/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/tf/notebooks/CKD/vision_transformer.py:370: UserWarning: Overwriting vit_small_patch16_224 in registry with vision_transformer.vit_small_patch16_224. This is because the name being registe

Hardware device registered: cuda


## Configuration — edit these

The block below is copied verbatim from the validation notebook (paths, laterality,
sign convention) so the gallery reproduces exactly the numbers you validated against.
**If you changed `IMAGE_LEFT_IS_SIDE` or `HKA_SIGN_FLIP` after reading that notebook's
laterality diagnostic, mirror the change here** — otherwise this gallery will be a
gallery of sign errors rather than of model failures.

Gallery-specific knobs follow underneath.

In [2]:
# ============================================================================
# From the validation notebook — keep in sync
# ============================================================================
OAI_IMAGE_DIR   = "/tf/data/hka/oai_dicoms"                # extracted <barcode>.dcm files
OAI_HKA_CSV     = "/tf/data/hka/oai_hka_groundtruth.csv"   # HKA ground truth
CHECKPOINT_PATH = "/tf/notebooks/kfolds_models/best_model_global.pt"

HKA_KEY_COL   = "barcode"        # == DICOM filename stem
HKA_SIDE1_COL = "hka_side1"
HKA_SIDE2_COL = "hka_side2"

HKA_NEUTRAL_180    = False       # OAISYS convention: signed, 0 = neutral, negative = varus
IMAGE_LEFT_IS_SIDE = "1"         # <- confirm against the validation notebook's diagnostic
HKA_SIGN_FLIP      = False       # <- ditto
DICOM_INVERT       = False
EXCLUDE_ABS_DEV    = 25.0        # |predicted HKA - neutral| beyond this => likely model failure

# ============================================================================
# Gallery configuration
# ============================================================================
# The two constants this notebook is built around. A knee's band is decided by
# |pred_hka - oai_hka| alone:
#
#     gross     |error| >  HKA_ERROR_THRESHOLD_DEG          (model failures)
#     moderate  HKA_ERROR_MID_DEG < |error| <= THRESHOLD    (drifting landmarks)
#     close     |error| <= HKA_ERROR_MID_DEG                (working as intended)
#
HKA_ERROR_THRESHOLD_DEG = 10.0
HKA_ERROR_MID_DEG       = 5.0

assert 0 < HKA_ERROR_MID_DEG < HKA_ERROR_THRESHOLD_DEG, "bands must be ordered and positive"

# Bands use the half-open rule  lo < |error| <= hi, so they are mutually exclusive and
# together cover every knee — no case is shown twice, none is invisible. The lowest band
# opens at -inf purely so that an exact 0.000 deg error still lands somewhere.
ERROR_BANDS = {
    "gross": dict(
        lo=HKA_ERROR_THRESHOLD_DEG, hi=float("inf"), accent="#ff2d55",
        title="Gross disagreement",
        label=f"|error| > {HKA_ERROR_THRESHOLD_DEG:.0f}°",
        blurb="Almost always a discrete, nameable fault — sign flip, mis-detected femoral "
              "head, cropped ankle, implant, out-of-distribution limb."),
    "moderate": dict(
        lo=HKA_ERROR_MID_DEG, hi=HKA_ERROR_THRESHOLD_DEG, accent="#ff9f0a",
        title="Moderate disagreement",
        label=f"{HKA_ERROR_MID_DEG:.0f}° < |error| ≤ {HKA_ERROR_THRESHOLD_DEG:.0f}°",
        blurb="The diagnostically interesting middle: landmarks are on the right bones but "
              "drifting. Clinically material (crosses varus/valgus categories) yet subtle "
              "enough that only the overlay shows which point moved."),
    "close": dict(
        lo=float("-inf"), hi=HKA_ERROR_MID_DEG, accent="#30d158",
        title="Close agreement",
        label=f"|error| ≤ {HKA_ERROR_MID_DEG:.0f}°",
        blurb="The bulk of the cohort. Browse it to confirm the model is right for the right "
              "reasons rather than by cancelling errors — and to sanity-check the knee-centre "
              "definition against OAISYS."),
}

INCLUDE_FLAGGED = True    # keep |HKA-neutral| > EXCLUDE_ABS_DEV knees — they are the
                          # gross failures, and excluding them hides the worst cases
MAX_IMAGES      = None    # None = whole cohort; set an int for a quick smoke test
RESULTS_CSV     = "hka_error_gallery_results.csv"   # per-knee cache written after the sweep
REUSE_CACHE     = True    # skip the sweep if RESULTS_CSV + landmark cache already exist
COORDS_NPZ      = "hka_error_gallery_coords.npz"    # predicted landmarks, keyed by barcode

GALLERY_ROWS = 2          # thumbnails per page
GALLERY_COLS = 3
THUMB_PX     = 300        # thumbnail render size
DETAIL_PX    = 820        # full-size render size

for _k, _b in ERROR_BANDS.items():
    print(f"{_k:9s} {_b['label']}")

gross     |error| > 10°
moderate  5° < |error| ≤ 10°
close     |error| ≤ 5°


## Image loading & preprocessing

Identical to the validation notebook: robust 1–99 percentile windowing, MONOCHROME1
handled automatically, letterbox to a square canvas so aspect ratio (and therefore every
angle) is preserved.

In [3]:
def preprocess_global_image(img, target_size=512):
    """Letterbox-resize *img* to a square canvas of *target_size* pixels."""
    orig_w, orig_h = img.size
    scale   = min(target_size / orig_w, target_size / orig_h)
    new_w   = int(orig_w * scale)
    new_h   = int(orig_h * scale)
    resized = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    pad_left = (target_size - new_w) // 2
    pad_top  = (target_size - new_h) // 2
    final_img = Image.new("RGB", (target_size, target_size), (0, 0, 0))
    final_img.paste(resized, (pad_left, pad_top))
    return final_img, scale, (pad_left, pad_top)


def load_oai_image(path):
    """Load an OAI radiograph as an RGB PIL image (handles DICOM windowing / inversion)."""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".dcm", ".dicom", ""):
        import pydicom
        ds  = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
        slope = ds.get("RescaleSlope", 1.0);     slope = 1.0 if slope in (None, "") else float(slope)
        inter = ds.get("RescaleIntercept", 0.0); inter = 0.0 if inter in (None, "") else float(inter)
        arr = arr * slope + inter
        lo, hi = np.percentile(arr, (1, 99))                           # robust window
        arr = np.clip((arr - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
        if ds.get("PhotometricInterpretation", "") == "MONOCHROME1" or DICOM_INVERT:
            arr = 1.0 - arr
        return Image.fromarray((arr * 255).astype(np.uint8)).convert("RGB")
    return Image.open(path).convert("RGB")


_IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def to_model_tensor(pil):
    t = torch.from_numpy(np.array(pil)).permute(2, 0, 1).float() / 255.0
    return (t - _IMNET_MEAN) / _IMNET_STD

## HKA geometry

`hka_from_side` is unchanged from the validation notebook — same sign convention, same
laterality reference — so an error computed here is the same number that appeared in the
Bland–Altman plot there. `hemisphere_for_side` is the only addition: it resolves which
half of the image an OAI *side* label refers to, which is what the renderer needs in order
to draw on the correct leg.

Slots within each hemisphere: `0` femur_head · `1` knee_inner · `2` ost_point ·
`3` knee_outer · `4` ankle_inner · `5` ankle_outer (`ost_point` is unused for HKA).

In [4]:
HEMI_SLOTS = {"left": slice(0, 6), "right": slice(6, 12)}
HEMI_NAMES = {"left": GLOBAL_KEYPOINT_NAMES[0:6], "right": GLOBAL_KEYPOINT_NAMES[6:12]}


def hka_from_side(pts6, neutral180=None, sign_flip=None):
    if neutral180 is None: neutral180 = HKA_NEUTRAL_180
    if sign_flip is None:  sign_flip  = HKA_SIGN_FLIP
    fh = pts6[0]; ki = pts6[1]; ko = pts6[3]
    kc = 0.5 * (ki + ko); ac = 0.5 * (pts6[4] + pts6[5])
    vf = fh - kc; vt = ac - kc                                  # knee->hip, knee->ankle
    cross = vf[0]*vt[1] - vf[1]*vt[0]; dot = vf[0]*vt[0] + vf[1]*vt[1]
    ang = abs(math.degrees(math.atan2(cross, dot)))             # ~180 for a straight leg
    dev = 180.0 - ang
    # laterality-consistent sign: knee_outer(lateral) - knee_inner(medial) is THIS leg's
    # medial->lateral direction, so the same clinical deformity gets the same sign on the
    # left and right legs.
    lat  = 1.0 if (ko[0] - ki[0]) >= 0 else -1.0
    sign = (-1.0 if cross < 0 else 1.0) * lat * (-1.0 if sign_flip else 1.0)
    return (180.0 - sign*dev) if neutral180 else (sign*dev)


def hemisphere_for_side(side):
    """OAI side label ('1'/'2') -> which half of the image it occupies."""
    side = str(side)
    if IMAGE_LEFT_IS_SIDE == "1":
        return "left" if side == "1" else "right"
    return "right" if side == "1" else "left"


def mech_axis_points(pts6):
    """(femoral head, knee centre, ankle centre) for one leg, in canvas coords."""
    fh = np.asarray(pts6[0], float)
    kc = 0.5 * (np.asarray(pts6[1], float) + np.asarray(pts6[3], float))
    ac = 0.5 * (np.asarray(pts6[4], float) + np.asarray(pts6[5], float))
    return fh, kc, ac


def predict_landmarks(pil):
    """OAI image -> [12,2] landmark coords in the 768 letterbox space (angle-preserving)."""
    canon, _, _ = preprocess_global_image(pil, TARGET_SIZE)
    with torch.no_grad():
        hms = torch.sigmoid(model_global(to_model_tensor(canon).unsqueeze(0).to(device)))
    return extract_coordinates(hms.cpu(), scale_factor=1.0 / HEATMAP_SCALE)[0].numpy()


# ---- self-tests: geometry must match the validation notebook exactly ----
_straight = np.array([[100,0],[90,500],[0,0],[110,500],[90,1000],[110,1000]], float)
assert abs(hka_from_side(_straight, neutral180=True, sign_flip=False) - 180.0) < 0.1, "HKA self-test failed"
_vL = np.array([[400,100],[415,500],[0,0],[385,500],[450,900],[430,900]], float)  # image-left,  varus
_vR = np.array([[400,100],[385,500],[0,0],[415,500],[350,900],[370,900]], float)  # image-right, mirror varus
assert abs(hka_from_side(_vL) - hka_from_side(_vR)) < 1e-6, "left/right sign inconsistent"
assert {hemisphere_for_side("1"), hemisphere_for_side("2")} == {"left", "right"}, "side mapping collapsed"
print("HKA self-test passed (straight -> 180; mirror legs share sign; side<->hemisphere is a bijection).")

HKA self-test passed (straight -> 180; mirror legs share sign; side<->hemisphere is a bijection).


## Model & checkpoint

In [5]:
_model_map = {
    "tiny":      Conformer_tiny_patch16_keypoint_half_heatmap,
    "small_p16": Conformer_small_patch16_keypoint_half_heatmap,
    "small_p32": Conformer_small_patch32_keypoint_half_heatmap,
    "base":      Conformer_base_patch16_keypoint_half_heatmap,
}

model_global = _model_map[MODEL_VARIANT](num_keypoints=12).to(device)

if os.path.exists(CHECKPOINT_PATH):
    model_global.load_state_dict(torch.load(CHECKPOINT_PATH, weights_only=True, map_location=device))
    model_global.eval()
    print(f"Loaded checkpoint '{CHECKPOINT_PATH}'.")
else:
    print(f"Checkpoint '{CHECKPOINT_PATH}' not found — set CHECKPOINT_PATH.")

Loaded checkpoint '/tf/notebooks/kfolds_models/best_model_global.pt'.


## Load OAI HKA ground truth

In [6]:
hka_lookup = None
if os.path.exists(OAI_HKA_CSV):
    hka_df = pd.read_csv(OAI_HKA_CSV, dtype={HKA_KEY_COL: str})
    hka_lookup = hka_df.set_index(HKA_KEY_COL)
    nboth = int(hka_df[[HKA_SIDE1_COL, HKA_SIDE2_COL]].notna().all(axis=1).sum())
    print(f"Loaded {len(hka_df)} full-limb images ({nboth} with both knees).")
    if {"readsd_side1", "readsd_side2"}.issubset(hka_df.columns):
        oai_reader_sd = pd.concat([hka_df["readsd_side1"], hka_df["readsd_side2"]]).dropna().mean()
        print(f"OAI within-knee reader agreement (SD): {oai_reader_sd:.3f} deg  <- benchmark")
else:
    print(f"Ground-truth CSV not found at '{OAI_HKA_CSV}' — run prepare_oai_hka.py first.")

Loaded 3831 full-limb images (3816 with both knees).
OAI within-knee reader agreement (SD): 0.298 deg  <- benchmark


## Inference sweep

One pass over the cohort. Unlike the validation notebook this **retains the predicted
landmark coordinates** for every image, because the gallery has to draw them. Coordinates
are 12×2 float32 per image — a few hundred KB for the whole cohort, so keeping them all is
cheaper than re-running inference on the subset later.

Results are cached to `RESULTS_CSV` + `COORDS_NPZ`; with `REUSE_CACHE = True` a second run
loads them and skips inference entirely. Delete those two files (or set `REUSE_CACHE = False`)
after changing the checkpoint or the sign convention.

In [7]:
IMG_EXTS = (".dcm", ".dicom", ".png", ".jpg", ".jpeg", ".tif", ".tiff")
NEUTRAL  = 180.0 if HKA_NEUTRAL_180 else 0.0

knees_df          = None
coords_by_barcode = {}
path_by_barcode   = {}

_cache_ready = REUSE_CACHE and os.path.exists(RESULTS_CSV) and os.path.exists(COORDS_NPZ)

if _cache_ready:
    knees_df = pd.read_csv(RESULTS_CSV, dtype={"barcode": str, "side": str})
    with np.load(COORDS_NPZ) as z:
        coords_by_barcode = {k: z[k].astype(np.float32) for k in z.files}
    path_by_barcode = dict(zip(knees_df["barcode"], knees_df["path"]))
    print(f"Reusing cache: {len(knees_df)} knees / {len(coords_by_barcode)} images "
          f"from '{RESULTS_CSV}'. Set REUSE_CACHE=False to force a fresh sweep.")

elif os.path.isdir(OAI_IMAGE_DIR) and os.path.exists(CHECKPOINT_PATH) and hka_lookup is not None:
    paths = sorted(p for p in glob.glob(os.path.join(OAI_IMAGE_DIR, "*"))
                   if os.path.splitext(p)[1].lower() in IMG_EXTS)
    if MAX_IMAGES is not None:
        paths = paths[:MAX_IMAGES]
    print(f"Found {len(paths)} images; matching to ground truth by barcode (filename stem)...")

    model_global.eval()
    records = []
    n_total = n_failed = 0

    try:
        from tqdm.auto import tqdm
        pbar = tqdm(paths, desc="OAI inference", unit="img"); use_tqdm = True
    except Exception:
        pbar = paths; use_tqdm = False

    for i, path in enumerate(pbar):
        stem = os.path.splitext(os.path.basename(path))[0]
        if stem not in hka_lookup.index:
            continue
        n_total += 1
        row = hka_lookup.loc[stem]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        try:
            coords = predict_landmarks(load_oai_image(path))            # [12,2], 768 space
        except Exception:
            n_failed += 1; continue

        coords_by_barcode[stem] = coords.astype(np.float32)
        path_by_barcode[stem]   = path

        p_left  = hka_from_side(coords[0:6])                            # image-left leg
        p_right = hka_from_side(coords[6:12])                           # image-right leg
        pred_side = ({"1": p_left, "2": p_right} if IMAGE_LEFT_IS_SIDE == "1"
                     else {"1": p_right, "2": p_left})

        g1 = pd.to_numeric(row.get(HKA_SIDE1_COL), errors="coerce")
        g2 = pd.to_numeric(row.get(HKA_SIDE2_COL), errors="coerce")

        for side, gt in (("1", g1), ("2", g2)):
            if pd.isna(gt):
                continue
            pred = float(pred_side[side]); gt = float(gt)
            records.append(dict(
                barcode=stem, side=side, hemisphere=hemisphere_for_side(side),
                pred_hka=pred, oai_hka=gt,
                signed_err=pred - gt, abs_err=abs(pred - gt),
                flagged=bool(abs(pred - NEUTRAL) > EXCLUDE_ABS_DEV),
                path=path,
            ))

        if use_tqdm:
            pbar.set_postfix(matched=n_total, knees=len(records), failed=n_failed)
        elif i % 200 == 0:
            print(f"  {i}/{len(paths)} images | matched {n_total}, knees {len(records)}, "
                  f"failed {n_failed}", flush=True)

    knees_df = pd.DataFrame(records)
    print(f"\nMatched {len(knees_df)} knees across {n_total} images ({n_failed} inference failures).")

    if len(knees_df):
        knees_df.to_csv(RESULTS_CSV, index=False)
        np.savez_compressed(COORDS_NPZ, **coords_by_barcode)
        print(f"Cached -> '{RESULTS_CSV}' and '{COORDS_NPZ}'.")

else:
    print("Set OAI_IMAGE_DIR / OAI_HKA_CSV / CHECKPOINT_PATH and run the cells above first.")

Found 3831 images; matching to ground truth by barcode (filename stem)...


OAI inference:   0%|          | 0/3831 [00:00<?, ?img/s]


Matched 7647 knees across 3831 images (0 inference failures).
Cached -> 'hka_error_gallery_results.csv' and 'hka_error_gallery_coords.npz'.


## Split the cohort into error bands

`select_error_band` is the single place the constants are applied — every gallery, every
slider, and the verification cell all route through it, so what appears on screen is always
exactly this rule and the three galleries can never drift apart.

In [8]:
def select_error_band(df, lo, hi, include_flagged=INCLUDE_FLAGGED, how="abs_desc", seed=0):
    """Knees with lo < |pred - OAI| <= hi, sorted by *how*."""
    sel = df[(df["abs_err"] > lo) & (df["abs_err"] <= hi)]
    if not include_flagged:
        sel = sel[~sel["flagged"].astype(bool)]
    return sort_selection(sel, how, seed)


def sort_selection(sel, how="abs_desc", seed=0):
    if   how == "abs_desc":    sel = sel.sort_values("abs_err", ascending=False)
    elif how == "abs_asc":     sel = sel.sort_values("abs_err", ascending=True)
    elif how == "signed_desc": sel = sel.sort_values("signed_err", ascending=False)
    elif how == "signed_asc":  sel = sel.sort_values("signed_err", ascending=True)
    elif how == "random":      sel = sel.sample(frac=1.0, random_state=seed)
    else:                      sel = sel.sort_values(["barcode", "side"])
    return sel.reset_index(drop=True)


def select_error_cases(df, threshold=HKA_ERROR_THRESHOLD_DEG, include_flagged=INCLUDE_FLAGGED):
    """Back-compatible helper: everything above *threshold* (the 'gross' band)."""
    return select_error_band(df, threshold, float("inf"), include_flagged)


def band_frame(name, df=None):
    df = knees_df if df is None else df
    b  = ERROR_BANDS[name]
    return select_error_band(df, b["lo"], b["hi"])


BANDS_DF   = {}
gallery_df = pd.DataFrame()

if knees_df is not None and len(knees_df):
    BANDS_DF   = {k: band_frame(k) for k in ERROR_BANDS}
    gallery_df = BANDS_DF["gross"]          # kept for backward compatibility

    n_all = len(knees_df)
    print("=" * 84)
    print(f"HKA ERROR BANDS — {n_all} knees across {knees_df['barcode'].nunique()} images")
    print("=" * 84)
    print(f"{'band':10s} {'range':26s} {'knees':>7s} {'share':>8s} {'images':>7s} "
          f"{'median':>8s} {'over/under':>12s} {'flagged':>8s}")
    print("-" * 84)
    for k, b in ERROR_BANDS.items():
        s = BANDS_DF[k]
        if not len(s):
            print(f"{k:10s} {b['label']:26s} {'0':>7s}")
            continue
        over = int((s["signed_err"] > 0).sum())
        print(f"{k:10s} {b['label']:26s} {len(s):7d} {100.0*len(s)/n_all:7.2f}% "
              f"{s['barcode'].nunique():7d} {s['abs_err'].median():7.2f}° "
              f"{over:5d}/{len(s)-over:<6d} {int(s['flagged'].sum()):8d}")
    print("-" * 84)

    covered = sum(len(s) for s in BANDS_DF.values())
    print(f"bands cover {covered}/{n_all} knees "
          f"({'complete partition' if covered == n_all else 'GAP — check band bounds'})")

    if len(gallery_df):
        both = gallery_df.groupby("barcode").size()
        print(f"\n{int((both == 2).sum())} images contribute BOTH legs to the gross band "
              f"(points at the image, not either knee)")
        print("\nWorst 10:")
        cols = ["barcode", "side", "hemisphere", "pred_hka", "oai_hka", "signed_err", "flagged"]
        print(gallery_df.head(10)[cols].to_string(index=False,
              formatters={"pred_hka": "{:.1f}".format, "oai_hka": "{:.1f}".format,
                          "signed_err": "{:+.1f}".format}))
    else:
        print(f"\nNo knee exceeds {HKA_ERROR_THRESHOLD_DEG:.1f}° — "
              f"max |error| was {knees_df['abs_err'].max():.2f}°.")
else:
    print("No results to select from — run the inference sweep above.")

HKA ERROR BANDS — 7647 knees across 3831 images
band       range                        knees    share  images   median   over/under  flagged
------------------------------------------------------------------------------------
gross      |error| > 10°                  534    6.98%     386   83.23°   179/355         475
moderate   5° < |error| ≤ 10°             182    2.38%     176    6.06°    84/98            0
close      |error| ≤ 5°                  6931   90.64%    3643    0.90°  2904/4027          0
------------------------------------------------------------------------------------
bands cover 7647/7647 knees (complete partition)

148 images contribute BOTH legs to the gross band (points at the image, not either knee)

Worst 10:
    barcode side hemisphere pred_hka oai_hka signed_err  flagged
16600803504    1       left    175.9    -8.8     +184.7     True
16602962710    1       left   -179.6     4.4     -184.0     True
16602127409    1       left    179.6    -4.3     +183.9     T

## Renderer

Each case is drawn on its own letterboxed 768 px canvas:

- **mechanical axis** — solid hip→knee-centre and knee-centre→ankle-centre segments (the
  two vectors whose angle *is* the HKA), plus a dotted hip→ankle chord for reference;
  `×` marks the derived knee and ankle centres
- **landmarks** — circles coloured per `LANDMARK_COLORS`; the osteotomy hinge (`ost_point`)
  is drawn as a square because HKA is blind to it
- the offending leg is drawn bright; the contralateral leg is optional and dimmed

Renders are memoised on `(barcode, side, size, overlay options)` so paging back and forth
costs nothing, and decoded canvases are LRU-cached so a DICOM is read at most once.

In [9]:
@lru_cache(maxsize=64)     # three galleries page independently, so keep a few more canvases
def canvas_for(barcode):
    """Letterboxed 768 canvas — the exact space the predicted landmarks live in."""
    canon, _, _ = preprocess_global_image(load_oai_image(path_by_barcode[barcode]), TARGET_SIZE)
    return canon


PRIMARY_COLOR = "#ff2d55"
OTHER_COLOR   = "#9aa0a6"

# Figure size is fixed in inches and the *dpi* is varied, so overlay geometry scales with
# the render automatically. 8.0 also makes px/8 exact in binary floating point, so the PNG
# comes out at exactly px x px rather than px-1.
FIG_INCHES = 8.0


def _draw_leg(ax, pts6, names, primary, show_landmarks, show_axis, boost):
    fh, kc, ac = mech_axis_points(pts6)
    col  = PRIMARY_COLOR if primary else OTHER_COLOR
    alph = 0.95 if primary else 0.40
    lw   = (2.4 if primary else 1.3) * boost

    if show_axis and np.all(np.isfinite([fh, kc, ac])):
        ax.plot([fh[0], kc[0]], [fh[1], kc[1]], "-", color=col, lw=lw, alpha=alph, zorder=3)
        ax.plot([kc[0], ac[0]], [kc[1], ac[1]], "-", color=col, lw=lw, alpha=alph, zorder=3)
        ax.plot([fh[0], ac[0]], [fh[1], ac[1]], ":", color=col, lw=max(lw * 0.6, 0.7),
                alpha=alph * 0.7, zorder=3)
        for p in (kc, ac):
            ax.plot(p[0], p[1], "x", color=col, ms=(9 if primary else 6) * boost,
                    mew=(2.0 if primary else 1.2) * boost, alpha=alph, zorder=4)

    if show_landmarks:
        for name, p in zip(names, pts6):
            if not np.all(np.isfinite(p)) or p[0] < 0:
                continue
            marker = "s" if "ost_point" in name else "o"     # square = not used by HKA
            ax.plot(p[0], p[1], marker,
                    color=LANDMARK_COLORS.get(name, "yellow"),
                    ms=(6.5 if primary else 4.0) * boost,
                    mec="white", mew=(0.9 if primary else 0.4) * boost,
                    alpha=0.95 if primary else 0.45, zorder=5)


def render_case(row, px=DETAIL_PX, show_landmarks=True, show_axis=True,
                show_other_leg=False, annotate=False, legend=False):
    """Render one gallery row to PNG bytes (exactly px x px)."""
    barcode = row["barcode"]
    coords  = np.asarray(coords_by_barcode[barcode], dtype=float)
    hemi    = row["hemisphere"]
    # thumbnails get proportionally chunkier marks, or they vanish at 300 px
    boost   = max(1.0, (DETAIL_PX / float(px)) ** 0.4)

    fig = Figure(figsize=(FIG_INCHES, FIG_INCHES), dpi=px / FIG_INCHES)
    FigureCanvasAgg(fig)
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_axis_off()
    ax.imshow(canvas_for(barcode))
    ax.set_xlim(0, TARGET_SIZE); ax.set_ylim(TARGET_SIZE, 0)

    order = [hemi] + ([h for h in ("left", "right") if h != hemi] if show_other_leg else [])
    for h in order:
        _draw_leg(ax, coords[HEMI_SLOTS[h]], HEMI_NAMES[h], h == hemi,
                  show_landmarks, show_axis, boost)

    if annotate:
        txt = (f"{barcode}  ·  side {row['side']} ({hemi})\n"
               f"pred {row['pred_hka']:+.1f}°   OAI {row['oai_hka']:+.1f}°   "
               f"err {row['signed_err']:+.1f}°")
        ax.text(0.015, 0.985, txt, transform=ax.transAxes, ha="left", va="top",
                fontsize=10.5 * boost, color="white", family="monospace",
                bbox=dict(boxstyle="round,pad=0.4", fc="black", ec=PRIMARY_COLOR, alpha=0.72))

    if legend:
        handles = [Line2D([], [], color=PRIMARY_COLOR, lw=2.2, label="mechanical axis"),
                   Line2D([], [], color=PRIMARY_COLOR, lw=0, marker="x", mew=2,
                          label="knee / ankle centre")]
        handles += [Line2D([], [], lw=0, marker="s" if "ost_point" in n else "o",
                           color=LANDMARK_COLORS.get(n, "yellow"), mec="white", mew=0.8,
                           ms=6, label=n.rsplit("_", 1)[0])
                    for n in HEMI_NAMES[hemi]]
        ax.legend(handles=handles, loc="lower right", fontsize=8 * boost,
                  framealpha=0.72, facecolor="black", labelcolor="white", ncol=1)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", facecolor="black")
    return buf.getvalue()


_png_cache = {}

def render_cached(row, **kw):
    key = (row["barcode"], row["side"], kw.get("px", DETAIL_PX), kw.get("show_landmarks", True),
           kw.get("show_axis", True), kw.get("show_other_leg", False),
           kw.get("annotate", False), kw.get("legend", False))
    if key not in _png_cache:
        if len(_png_cache) > 700:
            _png_cache.clear()
        _png_cache[key] = render_case(row, **kw)
    return _png_cache[key]


if len(gallery_df):
    _png = render_cached(gallery_df.iloc[0], px=260)
    print(f"Renderer OK — worst case rendered to {len(_png) / 1024:.0f} KB of PNG.")

Renderer OK — worst case rendered to 18 KB of PNG.


## Gallery builder

One factory, three instances — the bands differ only in their bounds and accent colour, so
building them from a shared `build_gallery` keeps the behaviour identical across all three
and means a fix to the navigation applies everywhere at once.

Each gallery is a paged thumbnail grid; click a tile's caption button to open it full size.

| control | effect |
|---|---|
| **\|error\|** | a *range* slider. Opens on the band's own bounds but can be dragged anywhere, so you can zoom in on 7–8° without touching the constants or re-running inference. |
| **sort** | worst-first, best-first, signed error (separates over- from under-prediction), barcode, or a random sample |
| **shuffle** | reshuffles the random sample — the practical way to browse a band with thousands of cases |
| **page** | jump straight to a page number |
| **landmarks / mech. axis** | toggle each overlay independently |
| **show other leg** | draw the contralateral leg dimmed, for whole-image problems |
| **caption on image** | burn barcode / pred / OAI / error into the PNG (useful for screenshots) |
| **include flagged** | keep or drop the `\|pred − neutral\| > 25°` gross failures |

Thumbnails render lazily — only the six on the current page — so opening the `≤ 5°` band
with several thousand cases in it costs the same as opening the gross band.

If `ipywidgets` is unavailable, `static_sheet()` falls back to a matplotlib contact sheet.

In [10]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAVE_WIDGETS = True
except Exception as _e:
    HAVE_WIDGETS = False
    print(f"ipywidgets unavailable ({_e}) — build_gallery() will fall back to static sheets.")

PAGE_SIZE = GALLERY_ROWS * GALLERY_COLS
ERR_MAX   = float(np.ceil(knees_df["abs_err"].max())) if knees_df is not None and len(knees_df) else 45.0

SORT_OPTIONS = [("largest |error| first", "abs_desc"),
                ("smallest |error| first", "abs_asc"),
                ("most over-predicted first", "signed_desc"),
                ("most under-predicted first", "signed_asc"),
                ("random sample", "random"),
                ("barcode", "barcode")]


def _meta_html(row, rank, total, accent):
    return f"""
    <div style="font-family:monospace;font-size:13px;line-height:1.7;padding-left:14px">
      <b style="font-size:15px">case {rank + 1} of {total}</b><br>
      barcode &nbsp;<b>{row['barcode']}</b><br>
      side &nbsp;&nbsp;&nbsp;&nbsp;<b>{row['side']}</b> &nbsp;(image {row['hemisphere']} leg)<br>
      predicted HKA &nbsp;<b>{row['pred_hka']:+.2f}&deg;</b><br>
      OAI HKA &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<b>{row['oai_hka']:+.2f}&deg;</b><br>
      error &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
        <b style="color:{accent}">{row['signed_err']:+.2f}&deg;</b>
        ({'over' if row['signed_err'] > 0 else 'under'}-predicted)<br>
      gross-failure flag &nbsp;<b>{'yes' if row['flagged'] else 'no'}</b><br>
      <span style="color:#888">{row['path']}</span>
    </div>"""


def static_sheet(sel, title, n_show=12):
    """matplotlib fallback when ipywidgets is unavailable."""
    import matplotlib.pyplot as plt
    if not len(sel):
        print(f"{title}: nothing to show"); return
    n_show = min(len(sel), n_show)
    ncol   = GALLERY_COLS
    nrow   = int(np.ceil(n_show / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 4.2 * nrow), squeeze=False)
    flat = axes.ravel()
    for ax, (_, row) in zip(flat, sel.head(n_show).iterrows()):
        ax.imshow(Image.open(io.BytesIO(render_cached(row, px=420, annotate=True))))
        ax.set_axis_off()
    for ax in flat[n_show:]:
        ax.set_axis_off()
    fig.suptitle(f"{title} — showing {n_show} of {len(sel)}", fontsize=14)
    fig.tight_layout(); plt.show()


def build_gallery(band, df=None, rows=GALLERY_ROWS, cols=GALLERY_COLS, sort="abs_desc"):
    """Self-contained interactive gallery for one entry of ERROR_BANDS.

    Every instance owns its widgets, its selection and its paging state, so the three
    galleries below are fully independent — changing the sort in one leaves the others
    untouched. They share only the render cache.
    """
    b   = ERROR_BANDS[band]
    df  = knees_df if df is None else df
    acc = b["accent"]
    per_page = rows * cols

    if df is None or not len(df):
        return widgets.HTML("<i>Nothing to display — run the inference sweep first.</i>") \
               if HAVE_WIDGETS else None

    if not HAVE_WIDGETS:
        static_sheet(select_error_band(df, b["lo"], b["hi"], how=sort),
                     f"{b['title']} ({b['label']})")
        return None

    L = widgets.Layout
    lo0 = max(b["lo"], 0.0)
    hi0 = min(b["hi"], ERR_MAX)

    rng = widgets.FloatRangeSlider(value=[lo0, hi0], min=0.0, max=ERR_MAX, step=0.25,
                                   description="|error|", readout_format=".2f",
                                   continuous_update=False, layout=L(width="380px"))
    sort_dd = widgets.Dropdown(description="sort", options=SORT_OPTIONS, value=sort,
                               layout=L(width="290px"))
    shuffle = widgets.Button(description="shuffle", tooltip="reshuffle the random sample",
                             layout=L(width="90px"))
    cb_lm    = widgets.Checkbox(value=True,  description="landmarks",        indent=False, layout=L(width="140px"))
    cb_ax    = widgets.Checkbox(value=True,  description="mech. axis",       indent=False, layout=L(width="140px"))
    cb_other = widgets.Checkbox(value=False, description="show other leg",   indent=False, layout=L(width="160px"))
    cb_cap   = widgets.Checkbox(value=False, description="caption on image", indent=False, layout=L(width="175px"))
    cb_flag  = widgets.Checkbox(value=INCLUDE_FLAGGED, description="include flagged",
                                indent=False, layout=L(width="155px"))

    prev_btn = widgets.Button(description="< prev", layout=L(width="85px"))
    next_btn = widgets.Button(description="next >", layout=L(width="85px"))
    page_box = widgets.BoundedIntText(value=1, min=1, max=1, layout=L(width="80px"))
    page_lbl = widgets.HTML(layout=L(width="120px"))
    status   = widgets.HTML()

    grid = widgets.GridBox(layout=L(grid_template_columns=f"repeat({cols}, {THUMB_PX + 16}px)",
                                    grid_gap="10px 10px"))
    grid_box = widgets.VBox([widgets.HBox([prev_btn, page_box, page_lbl, next_btn]), grid])

    d_img  = widgets.Image(format="png", layout=L(width=f"{DETAIL_PX}px"))
    d_meta = widgets.HTML()
    d_back = widgets.Button(description="< back to grid", button_style="info", layout=L(width="150px"))
    d_prev = widgets.Button(description="< previous", layout=L(width="110px"))
    d_next = widgets.Button(description="next >",     layout=L(width="110px"))
    detail_box = widgets.VBox([widgets.HBox([d_back, d_prev, d_next]),
                               widgets.HBox([d_img, d_meta])])
    detail_box.layout.display = "none"

    S = {"sel": None, "page": 0, "detail": None, "seed": 0, "mute": False}

    def _bounds():
        """Slider ends that touch the extremes stay open, so the band rule is preserved."""
        v0, v1 = rng.value
        return (-np.inf if v0 <= 0 else float(v0),
                np.inf if v1 >= ERR_MAX else float(v1))

    def _opts(px):
        return dict(px=px, show_landmarks=cb_lm.value, show_axis=cb_ax.value,
                    show_other_leg=cb_other.value, annotate=cb_cap.value)

    def _open_detail(idx):
        S["detail"] = idx
        row = S["sel"].iloc[idx]
        d_img.value  = render_cached(row, legend=True, **_opts(DETAIL_PX))
        d_meta.value = _meta_html(row, idx, len(S["sel"]), acc)
        d_prev.disabled = idx == 0
        d_next.disabled = idx >= len(S["sel"]) - 1
        grid_box.layout.display   = "none"
        detail_box.layout.display = "flex"

    def _tile(idx, row):
        img = widgets.Image(value=render_cached(row, **_opts(THUMB_PX)), format="png",
                            layout=L(width=f"{THUMB_PX}px", height=f"{THUMB_PX}px",
                                     border=f"2px solid {acc}"))
        btn = widgets.Button(description=f"#{idx + 1}  {row['barcode']}  {row['signed_err']:+.1f}°",
                             tooltip="open full size", layout=L(width=f"{THUMB_PX + 4}px"))
        btn.style.font_weight = "bold"
        btn.on_click(lambda _b, i=idx: _open_detail(i))
        return widgets.VBox([img, btn], layout=L(align_items="center"))

    def _refresh():
        sel   = S["sel"]
        pages = max(1, int(np.ceil(len(sel) / per_page)))
        S["page"] = min(max(S["page"], 0), pages - 1)
        start = S["page"] * per_page
        chunk = sel.iloc[start:start + per_page]
        grid.children = tuple(_tile(start + k, r) for k, (_, r) in enumerate(chunk.iterrows()))

        S["mute"] = True
        page_box.max, page_box.value = pages, S["page"] + 1
        S["mute"] = False
        page_lbl.value = (f"<div style='padding-top:5px;font-family:monospace'>/ {pages}</div>")
        prev_btn.disabled = S["page"] == 0
        next_btn.disabled = S["page"] >= pages - 1

        n, v0, v1 = len(sel), rng.value[0], rng.value[1]
        hi_txt = "∞" if v1 >= ERR_MAX else f"{v1:.2f}&deg;"
        status.value = (
            f"<div style='font-family:monospace;font-size:13px;padding:4px 0'>"
            f"<b style='color:{acc}'>{n}</b> knees in {v0:.2f}&deg;–{hi_txt} "
            f"({100.0 * n / max(len(df), 1):.2f}% of {len(df)}) &nbsp;·&nbsp; "
            f"{sel['barcode'].nunique()} images &nbsp;·&nbsp; "
            f"median {sel['abs_err'].median():.2f}&deg; &nbsp;·&nbsp; "
            f"max {sel['abs_err'].max():.2f}&deg;</div>"
            if n else "<div style='font-family:monospace;color:#888'>no knees in this range</div>")

    def _reselect(_=None):
        lo, hi = _bounds()
        S["sel"]  = select_error_band(df, lo, hi, cb_flag.value, sort_dd.value, S["seed"])
        S["page"] = 0
        _refresh()

    def _rerender(_=None):
        if S["detail"] is not None and detail_box.layout.display != "none":
            _open_detail(S["detail"])
        _refresh()

    def _goto_page(change):
        if S["mute"]:
            return
        S["page"] = int(change["new"]) - 1
        _refresh()

    def _shuffle(_b):
        S["seed"] += 1
        if sort_dd.value != "random":
            sort_dd.value = "random"      # triggers _reselect
        else:
            _reselect()

    rng.observe(_reselect, names="value")
    cb_flag.observe(_reselect, names="value")
    sort_dd.observe(_reselect, names="value")
    for cb in (cb_lm, cb_ax, cb_other, cb_cap):
        cb.observe(_rerender, names="value")
    page_box.observe(_goto_page, names="value")
    prev_btn.on_click(lambda _b: (S.__setitem__("page", S["page"] - 1), _refresh()))
    next_btn.on_click(lambda _b: (S.__setitem__("page", S["page"] + 1), _refresh()))
    shuffle.on_click(_shuffle)
    d_back.on_click(lambda _b: (S.__setitem__("detail", None),
                                setattr(detail_box.layout, "display", "none"),
                                setattr(grid_box.layout, "display", "flex")))
    d_prev.on_click(lambda _b: _open_detail(max(S["detail"] - 1, 0)))
    d_next.on_click(lambda _b: _open_detail(min(S["detail"] + 1, len(S["sel"]) - 1)))

    _reselect()
    ui = widgets.VBox([
        widgets.HTML(f"<h3 style='margin:2px 0;border-left:6px solid {acc};padding-left:9px'>"
                     f"{b['title']} &nbsp;<span style='color:{acc}'>{b['label']}</span></h3>"
                     f"<div style='color:#666;font-size:12.5px;max-width:900px;"
                     f"padding-left:15px'>{b['blurb']}</div>"),
        widgets.HBox([rng, sort_dd, shuffle]),
        widgets.HBox([cb_lm, cb_ax, cb_other, cb_cap, cb_flag]),
        status, grid_box, detail_box,
    ])
    ui._state = S            # exposed for the verification cell
    return ui


print("build_gallery ready — three instances follow.")

build_gallery ready — three instances follow.


### Gallery 1 — gross disagreement (`|error| > HKA_ERROR_THRESHOLD_DEG`)

The failures. Expect a small number of cases with individually identifiable causes; work
through them and label each one in the export at the bottom.

In [11]:
gallery_gross = build_gallery("gross")
if gallery_gross is not None:
    display(gallery_gross)

### Gallery 2 — moderate disagreement (`HKA_ERROR_MID_DEG` to `HKA_ERROR_THRESHOLD_DEG`)

The band that matters clinically. A 5–10° error is large enough to move a knee across the
varus / neutral / valgus boundary and therefore to change an osteotomy plan, but small
enough that nothing looks obviously broken — the landmarks are on the right bones, just
drifting. Watch the hip and the ankle mortise; those two carry the longest lever arms.

In [12]:
gallery_moderate = build_gallery("moderate")
if gallery_moderate is not None:
    display(gallery_moderate)

### Gallery 3 — close agreement (`|error| ≤ HKA_ERROR_MID_DEG`)

The majority of the cohort, so browse it with **sort → random sample** and the **shuffle**
button rather than trying to page through it. Two things worth confirming here: that the
agreement comes from correctly placed landmarks rather than from two errors cancelling
along the axis, and that the residual bias you see in Bland–Altman is the knee-centre
definition (your plateau midpoint vs the OAISYS notch/spines) rather than anything moving.

In [13]:
gallery_close = build_gallery("close", sort="random")
if gallery_close is not None:
    display(gallery_close)

## Verification

Cheap invariants that catch the failure modes that would silently produce a *wrong but
plausible-looking* gallery: a threshold that isn't actually applied, an error column that
drifted from its own inputs, a side→hemisphere mapping that draws on the wrong leg, or a
cached coordinate array that no longer matches the HKA it was stored with.

In [14]:
def _keyset(df):
    return set(map(tuple, df[["barcode", "side"]].itertuples(index=False)))


def verify(df_all, bands=None):
    bands  = BANDS_DF if bands is None else bands
    checks = []
    def chk(name, ok, detail=""):
        checks.append((name, bool(ok), detail))

    chk("band constants ordered and positive",
        0 < HKA_ERROR_MID_DEG < HKA_ERROR_THRESHOLD_DEG,
        f"mid={HKA_ERROR_MID_DEG} threshold={HKA_ERROR_THRESHOLD_DEG}")

    if df_all is None or not len(df_all):
        chk("results present", False, "inference sweep produced no rows")
    else:
        # --- the error column is what it claims to be ---
        recomputed = (df_all["pred_hka"] - df_all["oai_hka"]).abs()
        chk("abs_err == |pred - OAI| for every knee",
            np.allclose(recomputed, df_all["abs_err"], atol=1e-6),
            f"max drift {float((recomputed - df_all['abs_err']).abs().max()):.2e}")

        chk("no duplicate (barcode, side) rows",
            not df_all.duplicated(["barcode", "side"]).any(),
            f"{int(df_all.duplicated(['barcode', 'side']).sum())} duplicates")

        chk("hemisphere matches IMAGE_LEFT_IS_SIDE for every row",
            bool((df_all["side"].astype(str).map(hemisphere_for_side) == df_all["hemisphere"]).all()),
            f"IMAGE_LEFT_IS_SIDE={IMAGE_LEFT_IS_SIDE!r}")

        # --- the three bands tile the cohort exactly ---
        keysets = {k: _keyset(v) for k, v in bands.items()}
        overlaps = {f"{a}&{b}": len(keysets[a] & keysets[b])
                    for i, a in enumerate(bands) for b in list(bands)[i + 1:]}
        chk("bands are mutually exclusive", all(v == 0 for v in overlaps.values()),
            ", ".join(f"{k}={v}" for k, v in overlaps.items()) or "n/a")

        union = set().union(*keysets.values()) if keysets else set()
        chk("bands cover every knee (no case invisible)", union == _keyset(df_all),
            f"{len(union)} banded / {len(df_all)} total")

        # --- each band obeys its own bounds ---
        for k, b in ERROR_BANDS.items():
            s = bands.get(k, pd.DataFrame())
            ok = len(s) == 0 or bool(((s["abs_err"] > b["lo"]) & (s["abs_err"] <= b["hi"])).all())
            rng_txt = (f"{s['abs_err'].min():.2f}-{s['abs_err'].max():.2f} deg"
                       if len(s) else "empty")
            chk(f"band '{k}' respects {b['label']}", ok, f"n={len(s)}, {rng_txt}")

        # --- the legacy single-threshold helper still agrees with the gross band ---
        chk("select_error_cases() == gross band",
            _keyset(select_error_cases(df_all)) == keysets.get("gross", set()),
            "back-compatible")

        # --- everything the galleries need is actually present ---
        shown = pd.concat([b for b in bands.values() if len(b)]) if any(len(b) for b in bands.values()) \
                else pd.DataFrame(columns=df_all.columns)
        if len(shown):
            bcs = shown["barcode"].unique()
            miss_c = [b for b in bcs if b not in coords_by_barcode]
            miss_p = [b for b in bcs if b not in path_by_barcode]
            chk("landmarks cached for every displayed case", not miss_c, f"{len(miss_c)} missing")
            chk("image path known for every displayed case", not miss_p, f"{len(miss_p)} missing")
            chk("cached coords are finite and shaped [12,2]",
                all(np.asarray(coords_by_barcode[b]).shape == (12, 2) and
                    np.isfinite(coords_by_barcode[b]).all() for b in bcs[:300]))

            # pred_hka regenerates from the cached coords — catches a stale .npz
            worst, n_s = 0.0, 0
            for k, s in bands.items():
                for _, r in s.head(15).iterrows():
                    pts = np.asarray(coords_by_barcode[r["barcode"]], float)[HEMI_SLOTS[r["hemisphere"]]]
                    worst = max(worst, abs(hka_from_side(pts) - r["pred_hka"])); n_s += 1
            chk("pred_hka reproducible from cached landmarks", worst < 1e-4,
                f"max drift {worst:.2e} deg over {n_s} cases")

            # the renderer works for a case drawn from each non-empty band
            bad = []
            for k, s in bands.items():
                if not len(s):
                    continue
                png = render_cached(s.iloc[0], px=180)
                if not (png[:8] == b"\x89PNG\r\n\x1a\n" and len(png) > 1000):
                    bad.append(k)
            chk("renderer emits a valid PNG for every band", not bad, f"failed: {bad}" if bad else "")

    width = max(len(n) for n, _, _ in checks)
    print("=" * (width + 30))
    for name, ok, detail in checks:
        print(f"{'PASS' if ok else 'FAIL'}  {name.ljust(width)}  {detail}")
    print("=" * (width + 30))
    failed = [n for n, ok, _ in checks if not ok]
    print("All checks passed." if not failed else f"{len(failed)} CHECK(S) FAILED: {failed}")
    return not failed


if knees_df is not None and len(knees_df):
    verify(knees_df)
else:
    print("Nothing to verify yet — run the inference sweep first.")

PASS  band constants ordered and positive                  mid=5.0 threshold=10.0
PASS  abs_err == |pred - OAI| for every knee               max drift 0.00e+00
PASS  no duplicate (barcode, side) rows                    0 duplicates
PASS  hemisphere matches IMAGE_LEFT_IS_SIDE for every row  IMAGE_LEFT_IS_SIDE='1'
PASS  bands are mutually exclusive                         gross&moderate=0, gross&close=0, moderate&close=0
PASS  bands cover every knee (no case invisible)           7647 banded / 7647 total
PASS  band 'gross' respects |error| > 10°                  n=534, 10.29-184.71 deg
PASS  band 'moderate' respects 5° < |error| ≤ 10°          n=182, 5.01-9.96 deg
PASS  band 'close' respects |error| ≤ 5°                   n=6931, 0.00-4.99 deg
PASS  select_error_cases() == gross band                   back-compatible
PASS  landmarks cached for every displayed case            0 missing
PASS  image path known for every displayed case            0 missing
PASS  cached coords are finite and s

## What to look for

Triage is band-specific — the same overlay means different things depending on how large
the disagreement is.

**Gross (> 10°) — expect a nameable cause on nearly every case:**

1. **Mechanical axis drawn across the wrong leg.** The overlay lands on one leg while the
   error stays large → `IMAGE_LEFT_IS_SIDE` is wrong for this acquisition, not the model.
   If it happens on *every* image, fix the constant and re-run.
2. **Signed error ≈ 2 × OAI HKA.** A sign flip, not a localisation failure. The validation
   notebook's magnitude-only correlation is the aggregate version of this.
3. **Femoral head off the acetabulum, or on the pelvis midline.** The hip is the longest
   lever arm in the axis; a few centimetres there is several degrees of HKA.
4. **Implants, casts, callipers, or a flexed/rotated limb.** Out of the training
   distribution; note them but don't read them as landmark error.
5. **Both knees of one image in the band.** Points at the image (rotation, exposure,
   duplication), not at either knee.

**Moderate (5–10°) — nothing looks broken, so compare against the close band:**

6. **Ankle landmarks near the image edge or on the tibial shaft rather than the mortise.**
   Cropped or partially imaged ankles are a data problem, not a model problem — worth
   excluding rather than training on.
7. **Knee centre displaced along the joint line.** `knee_inner`/`knee_outer` sitting on the
   femoral condyles instead of the plateau shifts the midpoint sideways; small in pixels,
   several degrees in HKA.
8. **A systematic direction.** If this band is mostly one sign, you are looking at the
   knee-centre *definition* gap (your plateau midpoint vs the OAISYS notch/spines), which is
   a calibration offset rather than a failure — and correctable.

**Close (≤ 5°) — confirm the model is right for the right reasons:**

9. **Landmarks visibly off but HKA still correct.** Two errors cancelling along the axis.
   Rare, but it means the HKA metric is flattering the landmark quality, and the osteotomy
   hinge / plateau-width points (which HKA is blind to) may be worse than the ICC implies.

Every band is a plain DataFrame in `BANDS_DF` — add a `cause` column as you review and the
export below gives you a per-band triage sheet.

In [15]:
# Export each band for manual review / triage.
if BANDS_DF:
    for _name, _sel in BANDS_DF.items():
        if not len(_sel):
            print(f"{_name:9s} empty — nothing to export"); continue
        _out = _sel.copy()
        # fill in while reviewing: sign_flip / hip / ankle / knee_centre / implant / ok / other
        _out["cause"] = ""
        _out["band"]  = _name
        _p = f"hka_band_{_name}.csv"
        _out.to_csv(_p, index=False)
        print(f"{_name:9s} {len(_out):6d} rows -> {_p}")

    _all = pd.concat([s.assign(band=k) for k, s in BANDS_DF.items() if len(s)])
    _all.to_csv("hka_bands_all.csv", index=False)
    print(f"\ncombined  {len(_all):6d} rows -> hka_bands_all.csv")
else:
    print("No cases to export.")

gross        534 rows -> hka_band_gross.csv
moderate     182 rows -> hka_band_moderate.csv
close       6931 rows -> hka_band_close.csv

combined    7647 rows -> hka_bands_all.csv
